In [2]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [3]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


# Daily Leads and Partners

## Problem Description

You are given a DataFrame `daily_sales` with the following columns:

- `date_id` (str): Date in `YYYY-MM-DD` format
- `make_name` (str): Name of the make/brand
- `lead_id` (int): Lead identifier
- `partner_id` (int): Partner identifier

**Task:**  
For each combination of `date_id` and `make_name`, count the number of **distinct** `lead_id`'s and **distinct** `partner_id`'s.

**Output:**  
Return a DataFrame with columns:
- `date_id`
- `make_name`
- `unique_leads` – count of distinct `lead_id` per group
- `unique_partners` – count of distinct `partner_id` per group

The result does not need to be sorted in any specific order (though in the sample it is sorted for readability).

---

## Example

**Input `daily_sales`:**

| date_id    | make_name | lead_id | partner_id |
|------------|-----------|---------|------------|
| 2020-12-08 | toyota    | 0       | 1          |
| 2020-12-08 | toyota    | 1       | 0          |
| 2020-12-08 | toyota    | 1       | 2          |
| 2020-12-07 | toyota    | 0       | 2          |
| 2020-12-07 | toyota    | 0       | 1          |
| 2020-12-08 | honda     | 1       | 2          |
| 2020-12-08 | honda     | 2       | 1          |
| 2020-12-07 | honda     | 3       | 1          |
| 2020-12-07 | honda     | 3       | 1          |   *(duplicate row)*

**Expected Output:**

| date_id    | make_name | unique_leads | unique_partners |
|------------|-----------|--------------|-----------------|
| 2020-12-07 | honda     | 1            | 1               |
| 2020-12-07 | toyota    | 1            | 2               |
| 2020-12-08 | honda     | 2            | 2               |
| 2020-12-08 | toyota    | 2            | 3               |


In [1]:
## PySpark Dataset Generation

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema
schema = StructType([
    StructField("date_id", StringType(), True),
    StructField("make_name", StringType(), True),
    StructField("lead_id", IntegerType(), True),
    StructField("partner_id", IntegerType(), True)
])

# Data rows (including the duplicate row)
data = [
    ("2020-12-08", "toyota", 0, 1),
    ("2020-12-08", "toyota", 1, 0),
    ("2020-12-08", "toyota", 1, 2),
    ("2020-12-07", "toyota", 0, 2),
    ("2020-12-07", "toyota", 0, 1),
    ("2020-12-08", "honda", 1, 2),
    ("2020-12-08", "honda", 2, 1),
    ("2020-12-07", "honda", 3, 1),
    ("2020-12-07", "honda", 3, 1)   # duplicate
]

# Create DataFrame
daily_sales = spark.createDataFrame(data, schema=schema)

daily_sales.show()

+----------+---------+-------+----------+
|   date_id|make_name|lead_id|partner_id|
+----------+---------+-------+----------+
|2020-12-08|   toyota|      0|         1|
|2020-12-08|   toyota|      1|         0|
|2020-12-08|   toyota|      1|         2|
|2020-12-07|   toyota|      0|         2|
|2020-12-07|   toyota|      0|         1|
|2020-12-08|    honda|      1|         2|
|2020-12-08|    honda|      2|         1|
|2020-12-07|    honda|      3|         1|
|2020-12-07|    honda|      3|         1|
+----------+---------+-------+----------+



# Using Spark SQL

In [5]:
daily_sales.createOrReplaceTempView("sales")

In [8]:
spark.sql("""SELECT 
    date_id, 
    make_name, 
    COUNT(DISTINCT lead_id) AS unique_leads, 
    COUNT(DISTINCT partner_id) AS unique_partners
FROM sales
GROUP BY date_id, make_name ORDER BY date_id ;""").show()

+----------+---------+------------+---------------+
|   date_id|make_name|unique_leads|unique_partners|
+----------+---------+------------+---------------+
|2020-12-07|   toyota|           1|              2|
|2020-12-07|    honda|           1|              1|
|2020-12-08|    honda|           2|              2|
|2020-12-08|   toyota|           2|              3|
+----------+---------+------------+---------------+



# Using PySpark

In [11]:
from pyspark.sql.functions import countDistinct,col

result = daily_sales.groupBy("date_id", "make_name").agg(
    countDistinct("lead_id").alias("unique_leads"),
    countDistinct("partner_id").alias("unique_partners")
).orderBy(col("date_id"))
result.show()

+----------+---------+------------+---------------+
|   date_id|make_name|unique_leads|unique_partners|
+----------+---------+------------+---------------+
|2020-12-07|   toyota|           1|              2|
|2020-12-07|    honda|           1|              1|
|2020-12-08|    honda|           2|              2|
|2020-12-08|   toyota|           2|              3|
+----------+---------+------------+---------------+

